In [0]:
dbutils.widgets.text("catalog", "shopstream_dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
SELECT status, COUNT(*) FROM shopstream_dev.bronze.orders GROUP BY status
ORDER BY 2 DESC;

In [0]:
%sql

SELECT
(SELECT COUNT(*) FROM (SELECT order_line_id FROM
shopstream_dev.bronze.orders
GROUP BY order_line_id HAVING COUNT(*) > 1)) AS
duplicated_line_ids,
(SELECT COUNT(*) FROM shopstream_dev.bronze.orders WHERE quantity <= 0) AS
bad_quantity_rows;

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.silver.orders AS
WITH ranked AS (
SELECT *, ROW_NUMBER() OVER (PARTITION BY order_line_id ORDER BY order_ts)
AS rn
FROM {catalog}.bronze.orders
)
SELECT order_line_id, order_id, customer_id, product_id,
CAST(quantity AS INT) AS quantity,
CAST(unit_price AS DOUBLE) AS unit_price,
ROUND(quantity * unit_price, 2) AS line_revenue,
CAST(order_ts AS TIMESTAMP) AS order_ts,
LOWER(status) AS status,
NULLIF(coupon_code, '') AS coupon_code
FROM ranked
WHERE rn = 1 AND quantity > 0
""")

In [0]:
spark.sql(f"DELETE FROM {catalog}.silver.orders WHERE status = 'cancelled'")
display(spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {catalog}.silver.orders) AS rows_now,
(SELECT COUNT(*) FROM {catalog}.silver.orders VERSION AS OF 0) AS
rows_before_delete
"""))

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.silver.products AS
SELECT product_id, product_name, category,
CAST(unit_price AS DOUBLE) AS unit_price,
CAST(unit_cost AS DOUBLE) AS unit_cost,
ROUND(unit_price - unit_cost, 2) AS unit_margin
FROM {catalog}.bronze.products
""")

In [0]:
%sql
DESCRIBE HISTORY shopstream_dev.silver.orders